# Is the agreement with the tree semantic, or inherited from pixels?

**Questions.** (5.1) Does an RDM's agreement with the tree survive once low-level (pixel) similarity is held fixed? (5.2) Are CLIP and WordNet the same semantic surface, or do they explain different parts of the tree?

**Why it matters.** `sem_km` is a pure semantic construct, but images of the same category also look alike (a bird on a white background resembles another bird more than a hammer). An RDM can therefore agree with the tree simply because it tracks pixels. Partialling out the pixel RDM asks how much of the agreement is left once that route is closed. The commonality analysis then asks whether the semantic predictors (CLIP, WordNet) and behavior explain the tree through the same variance or through different parts of it.

**RDMs.** `sem_km` (the tree, always the target here), `sem_wn`, `clip_pre`, `clip_post`, `sens_pre` (pixels, the low-level control), `behav_joint` (raw, ~13% of pairs unobserved) and its two SMACOF embeddings `behav_joint_mds_low` (d = 5) and `behav_joint_mds_high` (d = 20), both complete, chosen in `1_mds_dimensionality` without the tree. Every quantity uses the pairs observed in all RDMs involved. Pre-SHINE pixels (`sens_pre`) is the control throughout, as in the pre-registration. Behavior as the *target* of a decomposition is RQ3 (s1, `6_decomposition`) and is not done here.

**Run from this directory** with the RDMs built.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s2_semantic_hierarchy/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from analysis.s2_semantic_hierarchy import common as c
from analysis.s2_semantic_hierarchy import semantic_vs_lowlevel as svl
from analysis.utils import plotting

import plotly.io as pio
pio.renderers.default = "notebook_connected"  # figures load plotly.js once instead of embedding it

In [2]:
N_BOOT_PARTIAL = 1000
N_BOOT_COMMONALITY = 500  # lower: each draw refits all 2^k - 1 subset models

try:
    RDMS = c.load_rdm_set(c.FULL_SET)
except FileNotFoundError as err:
    print(f"NOTE: {err}\nFalling back to RAW_SET; the MDS embeddings are left out of 5.1 and 5.2 (behav_joint, which has NaNs, "
          f"is used in 5.1 only).")
    RDMS = c.load_rdm_set(c.RAW_SET)

KM = RDMS[c.REFERENCE]
print(f"RDMs: {', '.join(RDMS)}; {len(KM):,} pairs; behav_joint observed on "
      f"{(~np.isnan(RDMS[c.BEHAV])).mean():.1%} of them")

RDMs: sem_km, sem_wn, clip_pre, clip_post, sens_pre, sens_post, behav_joint, behav_joint_mds_low, behav_joint_mds_high; 262,450 pairs; behav_joint observed on 86.9% of them


## 5.1 Partial rank correlation with the tree, controlling for pixels

**Question.** Is the Spearman correlation of each RDM with `sem_km` still there after the pixel RDM's rank structure is regressed out of both? For the two behavioral RDMs two further controls are added: CLIP (is behavior's agreement with the tree just CLIP's?) and CLIP plus pixels jointly.

**Method.** All three variables are rank-transformed on the pairs observed in every RDM involved, the target and predictor ranks are residualised on the control ranks by least squares, and the residuals are correlated. "Raw" is the same correlation without controls on the same pairs, so raw vs partial differ only by the control. The partial is a Spearman-type statistic (Kendall tau-a has no residualisation analogue). CIs are 95% percentile intervals of an image bootstrap.

In [3]:
specs = svl.default_specs(list(RDMS))
partial = svl.partial_table(RDMS, specs, n_boot=N_BOOT_PARTIAL)
fmt = partial.assign(
    raw=lambda d: d.raw.map("{:.3f}".format) + d.apply(lambda r: f" [{r.raw_lo:.3f}, {r.raw_hi:.3f}]", axis=1),
    partial=lambda d: d.partial.map("{:.3f}".format) + d.apply(lambda r: f" [{r.partial_lo:.3f}, {r.partial_hi:.3f}]", axis=1),
    retained=lambda d: d.retained.map("{:.0%}".format),
)[["predictor", "controls", "raw", "partial", "retained"]]
print(f"Spearman rho with sem_km on pairs observed in every RDM involved; 95% image-bootstrap CI, n_boot={N_BOOT_PARTIAL}")
print(fmt.to_string(index=False))
svl.plot_partial(partial).show()

Spearman rho with sem_km on pairs observed in every RDM involved; 95% image-bootstrap CI, n_boot=1000
           predictor              controls                  raw              partial retained
            clip_pre            pixels pre 0.348 [0.314, 0.386] 0.338 [0.304, 0.375]      97%
           clip_post            pixels pre 0.310 [0.277, 0.349] 0.298 [0.266, 0.338]      96%
              sem_wn            pixels pre 0.286 [0.239, 0.333] 0.294 [0.246, 0.345]     103%
         behav_joint            pixels pre 0.351 [0.336, 0.368] 0.344 [0.329, 0.360]      98%
 behav_joint_mds_low            pixels pre 0.557 [0.533, 0.583] 0.553 [0.528, 0.579]      99%
behav_joint_mds_high            pixels pre 0.625 [0.601, 0.649] 0.623 [0.599, 0.647]     100%
         behav_joint              CLIP pre 0.351 [0.336, 0.368] 0.276 [0.261, 0.290]      79%
         behav_joint CLIP pre + pixels pre 0.351 [0.336, 0.368] 0.276 [0.261, 0.290]      79%
 behav_joint_mds_low              CLIP pre 0.557 [0.

**Pixels do not explain the agreement.** Holding pixels fixed barely moves any RDM's correlation with the tree: CLIP pre 0.348 to 0.338 (97% retained), CLIP post 0.310 to 0.298 (96%), WordNet 0.286 to 0.294 (slightly higher, within the CIs), raw behavior 0.351 to 0.344 (98%), and the embeddings 0.557 to 0.553 (d = 5) and 0.625 to 0.623 (d = 20). Raw and partial CIs overlap almost completely in every row, so pixel similarity carries little of what these RDMs share with the tree.

**CLIP explains part of behavior's agreement; pixels add nothing beyond CLIP.** Controlling for CLIP pre lowers raw behavior's correlation with the tree from 0.351 to 0.276 (79% retained; CIs [0.336, 0.368] and [0.261, 0.290] do not overlap), the low-d embedding's from 0.557 to 0.475 (85%) and the high-d one's from 0.625 to 0.555 (89%). Adding pixels on top of CLIP changes nothing (0.276, 0.477, 0.558). Most of behavior's tree-like structure is therefore not CLIP's, but a tenth to a fifth of it is.

**Reading the embedding rows.** The embeddings correlate more strongly with the tree than raw behavior, but they are complete and smoothed while raw behavior is evaluated on the 86.9% of pairs it observed; only the raw-to-partial change within a row is the point here.

## 5.2 CLIP vs WordNet: one semantic surface, or two?

**Question.** CLIP and WordNet both agree with the tree. Do they do so through the same variance of `sem_km`, or through different parts of it? And how much of the tree is explained by anything beyond them (pixels, behavior)?

**Method.** The target is `sem_km` (rank-transformed throughout); the behavioral decomposition with behavior as the target is RQ3 (s1, `6_decomposition`) and is not done here. First, the incremental R^2 of CLIP pre and WordNet alone: if their joint R^2 equals the sum of the single R^2, they are complementary; if it is smaller, they overlap. Then a commonality analysis on {CLIP pre, WordNet, pixels pre, behavioral embedding}, run once with each embedding (an embedding rather than raw behavior because it is complete): the full-model R^2 is partitioned into the **unique** part of each predictor (what is lost when it alone is dropped) and the **shared** parts (variance explained by exactly that subset of predictors, 2^4 - 1 = 15 components in all, summing to the full R^2). A shared component can be slightly negative when one predictor suppresses noise in another; those are reported as they are. CIs are image-bootstrap percentile intervals, with 500 draws here (instead of 1000) because each draw refits all subset models.

In [4]:
CLIP_WN = {k: RDMS[k] for k in ["clip_pre", "sem_wn"]}
inc = svl.incremental_r2(KM, CLIP_WN)
print(inc.round(4).to_string(index=False))

          predictors     r2
            clip_pre 0.1212
              sem_wn 0.0816
   clip_pre + sem_wn 0.1940
(sum if independent) 0.2029


**CLIP and WordNet are largely complementary.** Alone, CLIP pre explains 12.1% of the rank variance of the tree and WordNet 8.2%; together 19.4%, against 20.3% if they were independent. They share only about 0.9 percentage points: different semantic surfaces, each carrying its own part of the tree.

In [5]:
RESULTS = {}
for emb in [k for k in c.BEHAV_MDS_SET if k in RDMS] or [None]:
    predictors = ["clip_pre", "sem_wn", "sens_pre"] + ([emb] if emb else [])
    RESULTS[emb] = svl.commonality_with_ci(KM, {k: RDMS[k] for k in predictors}, n_boot=N_BOOT_COMMONALITY)

for emb, result in RESULTS.items():
    print(f"=== behavioral predictor: {emb} (target sem_km, rank-transformed; n_boot={N_BOOT_COMMONALITY}) ===")
    summary = result["summary"]
    print(pd.DataFrame({
        "unique R2 [95% CI]": [f"{r.unique:.4f} [{r.unique_lo:.4f}, {r.unique_hi:.4f}]" for r in summary.itertuples()],
        "shared with others": summary.shared_with_others.map("{:.4f}".format),
        "R2 alone": summary.r2_alone.map("{:.4f}".format),
    }, index=summary.index).to_string())
    lo, hi = result["shared_total_ci"]
    print(f"total shared (two or more predictors): {result['shared_total']:.4f} [{lo:.4f}, {hi:.4f}]")
    lo, hi = result["full_r2_ci"]
    print(f"full-model R2: {result['full_r2']:.4f} [{lo:.4f}, {hi:.4f}]\n")

=== behavioral predictor: behav_joint_mds_low (target sem_km, rank-transformed; n_boot=500) ===
                          unique R2 [95% CI] shared with others R2 alone
predictor                                                               
clip_pre             0.0110 [0.0053, 0.0201]             0.1102   0.1212
sem_wn               0.0354 [0.0215, 0.0544]             0.0462   0.0816
sens_pre             0.0002 [0.0000, 0.0023]             0.0079   0.0081
behav_joint_mds_low  0.1615 [0.1365, 0.1880]             0.1493   0.3108
total shared (two or more predictors): 0.1482 [0.1306, 0.1712]
full-model R2: 0.3564 [0.3311, 0.3875]

=== behavioral predictor: behav_joint_mds_high (target sem_km, rank-transformed; n_boot=500) ===
                           unique R2 [95% CI] shared with others R2 alone
predictor                                                                
clip_pre              0.0011 [0.0000, 0.0043]             0.1202   0.1212
sem_wn                0.0396 [0.0250, 0.0584

In [6]:
for emb, result in RESULTS.items():
    components = result["components"]
    print(f"=== all components, behavioral predictor {emb} ===")
    print(components.assign(value=components.value.round(4)).to_string(index=False))
    print(f"sum of components = {components.value.sum():.4f}; full-model R2 = {result['full_r2']:.4f}\n")
    svl.plot_unique(result).show()

=== all components, behavioral predictor behav_joint_mds_low ===
                                         component  order   kind   value
                                          clip_pre      1 unique  0.0110
                                            sem_wn      1 unique  0.0354
                                          sens_pre      1 unique  0.0002
                               behav_joint_mds_low      1 unique  0.1615
                                 clip_pre & sem_wn      2 shared -0.0011
                               clip_pre & sens_pre      2 shared -0.0002
                    clip_pre & behav_joint_mds_low      2 shared  0.0902
                                 sem_wn & sens_pre      2 shared  0.0009
                      sem_wn & behav_joint_mds_low      2 shared  0.0382
                    sens_pre & behav_joint_mds_low      2 shared  0.0007
                      clip_pre & sem_wn & sens_pre      3 shared -0.0006
           clip_pre & sem_wn & behav_joint_mds_low      3 s

=== all components, behavioral predictor behav_joint_mds_high ===
                                          component  order   kind   value
                                           clip_pre      1 unique  0.0011
                                             sem_wn      1 unique  0.0396
                                           sens_pre      1 unique  0.0012
                               behav_joint_mds_high      1 unique  0.2392
                                  clip_pre & sem_wn      2 shared -0.0002
                                clip_pre & sens_pre      2 shared -0.0003
                    clip_pre & behav_joint_mds_high      2 shared  0.1002
                                  sem_wn & sens_pre      2 shared  0.0018
                      sem_wn & behav_joint_mds_high      2 shared  0.0340
                    sens_pre & behav_joint_mds_high      2 shared -0.0003
                       clip_pre & sem_wn & sens_pre      3 shared -0.0001
           clip_pre & sem_wn & behav_joint_mds

**Adding behavior changes the picture, with either embedding.** With the high-d embedding (d = 20) the four predictors explain 43.4% [40.7, 46.5] of the tree's rank variance; 23.9% [20.8, 27.3] is unique to behavior, 4.0% [2.5, 5.8] to WordNet, and 15.3% [13.1, 18.2] is shared by two or more predictors. **CLIP's unique share shrinks to 0.1% [0.0, 0.4]**: its 12.1% alone is almost entirely shared, mostly with behavior alone (the CLIP and behavior component is 10.0%). Pixels add 0.1% [0.01, 0.4] uniquely. WordNet keeps a clear unique part and shares 3.4% with behavior alone.

With the low-d embedding (d = 5) the pattern is the same at a lower level: full R^2 35.6% [33.1, 38.8], unique to behavior 16.2% [13.7, 18.8], to WordNet 3.5% [2.2, 5.4], to CLIP 1.1% [0.5, 2.0], to pixels 0.02%; shared 14.8% [13.1, 17.1], of which CLIP and behavior 9.0%. The low-d embedding leaves CLIP a small unique part that the high-d one absorbs.

So behavior covers nearly everything CLIP knows about the tree, plus a large part that neither CLIP nor WordNet has, while WordNet's contribution is only partly captured by behavior. Pixels add nothing in any combination, in line with 5.1; the negative shared terms (down to -0.30%) are suppression within the noise of a few tenths of a percent. The components sum exactly to the full R^2 in both models.

**Caveat on reading "unique".** These decompose predictive overlap, not causes: a predictor's unique share depends on which others are in the model, and the CLIP and behavior overlap is symmetric (CLIP could equally be said to explain behavior). R^2 is computed over 262,450 non-independent pairs, so only the image-bootstrap CIs are interpretable, not analytic tests.

## Takeaways

1. **The agreement with the tree is not inherited from pixels.** Controlling for pixel similarity keeps 96% to 100% of every RDM's correlation with the tree: CLIP pre 0.348 to 0.338, CLIP post 0.310 to 0.298, raw behavior 0.351 to 0.344, embeddings 0.557 to 0.553 (d = 5) and 0.625 to 0.623 (d = 20); WordNet is unchanged (0.286 to 0.294).
2. **CLIP explains some of behavior's agreement with the tree, pixels none beyond that.** Controlling for CLIP pre leaves 79% of raw behavior's correlation (0.351 to 0.276), 85% of the low-d embedding's (0.557 to 0.475) and 89% of the high-d one's (0.625 to 0.555); adding pixels as a second control changes none of them.
3. **CLIP and WordNet are largely different semantic surfaces.** Jointly they explain 19.4% of the tree's rank variance, against 12.1% (CLIP) and 8.2% (WordNet) alone and 20.3% if they were independent.
4. **Behavior absorbs CLIP's contribution but not WordNet's.** With the high-d embedding (full R^2 0.434 [0.407, 0.465]) the unique parts are behavior 0.239 [0.208, 0.273], WordNet 0.040 [0.025, 0.058], CLIP 0.001 [0.000, 0.004] and pixels 0.001 [0.000, 0.004], with 0.153 [0.131, 0.182] shared. With the low-d embedding (full R^2 0.356 [0.331, 0.388]): behavior 0.162 [0.137, 0.188], WordNet 0.035 [0.022, 0.054], CLIP 0.011 [0.005, 0.020], pixels 0.000.

**Limitations.** Pixels are the pre-SHINE pixel RDM only, and the partial removes only their monotone (rank-linear) contribution; another low-level feature (edges, colour) could still carry some of the agreement. `sem_km` has only 9 distinct values, so rank statistics on it are tie-heavy. The embeddings are smoothed and complete, so their rows are not directly comparable with raw behavior's. Commonality components are specific to the predictor set chosen. Behavior as the *target* of this decomposition (RQ3) is not addressed here.